# NASA FIRMS · Fire Detections in Spain, 2012–2025

The catalog contains all available point detections from **VIIRS Suomi‑NPP, NOAA‑20, and NOAA‑21** for the selected period. **MODIS Aqua** is also enabled because the current model uses it for historical features. The area is the same as in the previous catalog: Spain within the Iberian bounding box, excluding the Canary Islands.

1. Run cell 1 and allow Colab to connect to Google Drive. This creates the `MyDrive/spain_firms_2012_2025/` folder.
2. Run cell 2. Set `FIRMS_MAP_KEY` in Colab Secrets or enter it at the hidden prompt. The cell shows the available dates and sources. You can get a key from [NASA FIRMS](https://firms.modaps.eosdis.nasa.gov/api/map_key/).
3. Run cell 3. It checks a sample from each source, then downloads the remaining date windows. If the run is interrupted, rerun cells 1–3 to resume from the cache. If FIRMS returns HTTP 429, wait for the key's rate limit to reset, then rerun cell 3.
4. Run cell 4. The result is saved to `detections/detections_spain.parquet`.

Standard Processing (SP) takes priority. For dates when only Near Real-Time (NRT) data are available, the catalog records the product in `source_product` and leaves missing `type` values blank. Dates for which the API offers no product are listed in `firms_download_plan.csv`. The number of records with a known type by year and platform is listed in `firms_coverage_by_year.csv`.

The original CSV responses are stored in `cache/`. The Parquet file preserves all fields returned by the API, including `type`, `confidence`, `version`, `satellite`, `instrument`, `daynight`, `scan`, `track`, `bright_ti4`, `bright_ti5`, and `frp`. Records are kept regardless of type or confidence. FIRMS classifies `type = 0` as a presumed vegetation fire.


In [ ]:
# 1. Mount Drive and select the historical period.
from datetime import date
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/spain_firms_2012_2025")
CACHE = ROOT / "cache"
OUTPUT = ROOT / "detections"
OUTPUT.mkdir(parents=True, exist_ok=True)

FIRST, LAST = date(2012, 1, 1), date(2025, 12, 31)
BBOX = (-10.0, 35.5, 4.5, 44.0)  # west, south, east, north
INCLUDE_AQUA = True  # Used by the existing forecasting workflow.
WORKERS = 4

print("Google Drive folder:", ROOT)


In [ ]:
# 2. Check the date ranges for each FIRMS product.
import json
import re
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta, timezone
from getpass import getpass
from hashlib import sha256
from html import unescape
from http.client import IncompleteRead
from importlib.util import find_spec
from io import BytesIO
from time import sleep
from urllib.error import HTTPError, URLError
from urllib.parse import quote, unquote
from urllib.request import urlopen
from uuid import uuid4

import pandas as pd
from google.colab import userdata
from IPython.display import display

missing = [name for name in ("shapely", "pyarrow") if find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
from shapely import contains_xy
from shapely.geometry import box, shape

FIRMS = "https://firms.modaps.eosdis.nasa.gov"
MASK_URL = (
    "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/v5.1.2/"
    "geojson/ne_10m_admin_0_countries.geojson"
)
MASK_SHA256 = "239eec57ac17f100a11e2536cffc56752c318b50ae765b0918ff7aab4ce8f255"
# The published first date may differ from the span available through a specific API product.
SOURCES = {
    "SNPP": ("VIIRS_SNPP", date(2012, 1, 20)),
    "NOAA20": ("VIIRS_NOAA20", date(2018, 4, 1)),
    "NOAA21": ("VIIRS_NOAA21", date(2023, 3, 1)),
}
if INCLUDE_AQUA:
    SOURCES["Aqua"] = ("MODIS", date(2002, 7, 4))

def atomic_path_write(path, write):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid4().hex}.tmp")
    try:
        write(temporary)
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)

class FirmsHTTPError(RuntimeError):
    def __init__(self, status, label, detail=""):
        self.status = status
        self.detail = detail
        reason = f"HTTP {status}" + (f": {detail}" if detail else "")
        super().__init__(f"FIRMS download failed for {label}: {reason}. Rerun to resume.")

def http_error_detail(error):
    try:
        detail = error.read(4096).decode("utf-8", errors="replace")
    except (OSError, IncompleteRead):
        return ""
    finally:
        error.close()
    detail = unescape(detail)
    for secret in {KEY, unquote(KEY)}:
        if secret:
            detail = detail.replace(secret, "[REDACTED]")
    detail = re.sub(r"(?i)https?://[^\s<>]+", "[URL omitted]", detail)
    detail = re.sub(r"<[^>]*>", " ", detail)
    return " ".join(detail.split())[:240]

def fetch(url, label):
    for attempt in range(5):
        failure = None
        try:
            with urlopen(url, timeout=60) as response:
                payload = response.read()
                size = response.headers.get("Content-Length")
            if payload and (size is None or len(payload) == int(size)):
                return payload
            reason = "empty or incomplete response"
        except HTTPError as error:
            failure = FirmsHTTPError(error.code, label, http_error_detail(error))
            if error.code not in {408, 429, 500, 502, 503, 504}:
                raise failure from None
        except (URLError, TimeoutError, IncompleteRead, OSError):
            reason = "network error"
        if attempt < 4:
            sleep(2 ** (attempt + 1))
    if failure is not None:
        raise failure from None
    raise RuntimeError(f"FIRMS download failed for {label}: {reason}. Rerun to resume.") from None

def source_plan(availability):
    rows = []
    for platform, (prefix, published_first) in SOURCES.items():
        day = max(FIRST, published_first)
        products = [f"{prefix}_SP", f"{prefix}_NRT"]
        ranges = {key: availability[key] for key in products if key in availability}
        while day <= LAST:
            product = next((key for key in products if key in ranges
                            and ranges[key][0] <= day <= ranges[key][1]), None)
            # Split only when the selected product could change.
            boundaries = [LAST + timedelta(days=1)]
            for first, last in ranges.values():
                boundaries.extend(edge for edge in (first, last + timedelta(days=1)) if edge > day)
            stop = min(boundaries)
            if rows and rows[-1]["platform"] == platform and rows[-1]["source"] == product:
                rows[-1]["last"] = stop - timedelta(days=1)
            else:
                rows.append(dict(platform=platform, source=product, first=day, last=stop - timedelta(days=1)))
            day = stop
    return rows

try:
    key = userdata.get("FIRMS_MAP_KEY")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    key = ""
if not key or not key.strip():
    key = getpass("NASA FIRMS MAP_KEY: ")
if not key.strip():
    raise ValueError("Enter your NASA FIRMS MAP_KEY or add it to Colab Secrets.")
KEY = quote(key.strip(), safe="")
del key

payload = fetch(f"{FIRMS}/api/data_availability/csv/{KEY}/all", "data availability")
availability_table = pd.read_csv(BytesIO(payload), dtype="string")
expected_columns = {"data_id", "min_date", "max_date"}
if not expected_columns <= set(availability_table):
    raise RuntimeError("FIRMS returned an unexpected availability response. Check the MAP_KEY.")
availability = {}
for name, first, last in availability_table[["data_id", "min_date", "max_date"]].dropna().itertuples(index=False, name=None):
    availability[name.strip().upper()] = (date.fromisoformat(first.strip()), date.fromisoformat(last.strip()))
plan = source_plan(availability)
plan_table = pd.DataFrame(plan)
atomic_path_write(OUTPUT / "firms_download_plan.csv", lambda path: plan_table.to_csv(path, index=False))
display(plan_table.fillna("UNAVAILABLE"))
for row in plan:
    if not row["source"] or row["source"].endswith("_NRT"):
        note = "not available through this API" if not row["source"] else "NRT fallback, type may be missing"
        print(f"Coverage notice: {row['platform']} {row['first']}–{row['last']}: {note}")


In [ ]:
# 3. Fetch and cache the original FIRMS CSV files.
SATELLITES = {
    "SNPP": {"N", "NP", "NPP", "SNPP", "SUOMINPP"},
    "NOAA20": {"1", "N20", "J1", "J01", "JPSS1", "NOAA20"},
    "NOAA21": {"2", "N21", "J2", "J02", "JPSS2", "NOAA21"},
    "Aqua": {"A", "AQUA", "T", "TERRA"},
}
REQUIRED = {"latitude", "longitude", "acq_date", "acq_time", "satellite", "frp"}
ATTRIBUTES = ["type", "confidence", "version", "instrument", "daynight", "scan", "track",
              "bright_ti4", "bright_ti5", "brightness", "bright_t31"]
NUMERIC = ["frp", "scan", "track", "bright_ti4", "bright_ti5", "brightness", "bright_t31"]
DETECTION_KEY = ["time_utc", "platform", "lat", "lon"]
BBOX_ARG = ",".join(map(str, BBOX))

def read_mask():
    path = CACHE / "natural_earth_countries.geojson"
    payload = path.read_bytes() if path.is_file() else b""
    if sha256(payload).hexdigest() != MASK_SHA256:
        payload = fetch(MASK_URL, "Natural Earth Spain boundary")
        if sha256(payload).hexdigest() != MASK_SHA256:
            raise ValueError("Natural Earth boundary checksum mismatch.")
        atomic_path_write(path, lambda tmp: tmp.write_bytes(payload))
    feature = next(f for f in json.loads(payload)["features"] if f["properties"].get("NAME") == "Spain")
    return shape(feature["geometry"]).intersection(box(*BBOX))

def parse_csv(payload, platform, product, first, last, mask):
    if re.fullmatch(rb"no (?:data|detections|active fire detections)(?: found| available)?"
                    rb"(?: for (?:this|the) (?:area|date range|time period|location|request))?[.!]?",
                    payload.strip(), flags=re.IGNORECASE):
        return pd.DataFrame()
    data = pd.read_csv(BytesIO(payload), dtype="string")
    data.columns = data.columns.str.strip().str.lower()
    if not REQUIRED <= set(data):
        raise ValueError(f"Missing required FIRMS columns: {sorted(REQUIRED - set(data))}")
    if data.empty:
        return pd.DataFrame()

    satellite = data["satellite"].str.strip().str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)
    if not satellite.isin(SATELLITES[platform]).all():
        raise ValueError("Unexpected satellite in the response.")
    data["acq_time"] = data["acq_time"].str.strip().str.zfill(4)
    time_utc = pd.to_datetime(
        data["acq_date"].str.strip() + " " + data["acq_time"],
        format="%Y-%m-%d %H%M", utc=True, errors="coerce",
    )
    if not time_utc.between(pd.Timestamp(first, tz="UTC"),
                            pd.Timestamp(last + timedelta(days=1), tz="UTC"), inclusive="left").all():
        raise ValueError("Acquisition timestamp missing or outside the query dates.")
    lat = pd.to_numeric(data["latitude"], errors="coerce").astype("float64")
    lon = pd.to_numeric(data["longitude"], errors="coerce").astype("float64")
    if not (lat.between(-90, 90) & lon.between(-180, 180)).all():
        raise ValueError("Invalid coordinates in the response.")
    data = data.assign(time_utc=time_utc, lat=lat, lon=lon, latitude=lat, longitude=lon,
                       platform=platform, source_product=product)
    inside = contains_xy(mask, lon.to_numpy(), lat.to_numpy())
    if platform == "Aqua":
        inside &= satellite.isin({"A", "AQUA"}).to_numpy(dtype=bool)
    data = data.loc[inside].copy()
    for column in ATTRIBUTES:
        if column not in data:
            data[column] = pd.Series(pd.NA, index=data.index, dtype="string")
    for column in NUMERIC:
        values = pd.to_numeric(data[column], errors="coerce")
        data[column] = values.mask(values.isin([float("inf"), -float("inf")])).astype("float32")
    data["frp"] = data["frp"].where(data["frp"].ge(0))
    kinds = pd.to_numeric(data["type"], errors="coerce")
    data["type"] = kinds.where(kinds.isin([0, 1, 2, 3])).astype("Int8")
    return data

def get_daily_windows(task, mask):
    platform, product, first, last = task
    frames = []
    downloaded = False
    for offset in range((last - first).days + 1):
        day = first + timedelta(days=offset)
        data, fetched = get_window((platform, product, day, day), mask)
        if not data.empty:
            frames.append(data)
        downloaded |= fetched
    return (pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()), downloaded

def get_window(task, mask):
    platform, product, first, last = task
    days = (last - first).days + 1
    path = CACHE / product / str(first.year) / f"{first}_{days}d.csv"
    if path.is_file():
        try:
            return parse_csv(path.read_bytes(), *task, mask), False
        except (ValueError, OSError):
            pass
    url = f"{FIRMS}/api/area/csv/{KEY}/{product}/{BBOX_ARG}/{days}/{first}"
    label = f"{product} {first}–{last}"
    split_marker = path.with_suffix(".split")
    if days > 1 and split_marker.is_file():
        return get_daily_windows(task, mask)
    try:
        payload = fetch(url, label)
    except FirmsHTTPError as error:
        credential_or_quota_error = re.search(
            r"map.?key|api.?key|credential|authenticat|authoriz|forbidden|transaction|rate.?limit|quota|too many requests",
            error.detail, flags=re.IGNORECASE,
        )
        if error.status != 400 or days == 1 or credential_or_quota_error:
            raise
        print(f"{label}: HTTP 400. Retrying one day at a time.", flush=True)
        data, downloaded = get_daily_windows(task, mask)
        atomic_path_write(split_marker, lambda tmp: tmp.write_text("daily\n", encoding="utf-8"))
        return data, downloaded
    try:
        data = parse_csv(payload, *task, mask)
    except ValueError as error:
        raise RuntimeError(f"{label}: {error}. Correct the issue and rerun this cell.") from None
    atomic_path_write(path, lambda tmp: tmp.write_bytes(payload))
    return data, True

def year_tasks(row, year):
    first = max(row["first"], date(year, 1, 1))
    last = min(row["last"], date(year, 12, 31))
    for offset in range(0, (last - first).days + 1, 5):
        start = first + timedelta(days=offset)
        yield row["platform"], row["source"], start, min(start + timedelta(days=4), last)

mask = read_mask()
parts, coverage = [], []
download_complete = False
for row in plan:
    if row["source"]:
        sample = next(year_tasks(row, row["first"].year))
        data, _ = get_window(sample, mask)
        known = int(data["type"].notna().sum()) if not data.empty else 0
        print(f"Sample {row['source']} {sample[2]}: {len(data):,} rows, {known:,} with type")
        if not data.empty and known < len(data):
            print("  This response includes detections with unknown type.")

with ThreadPoolExecutor(max_workers=WORKERS) as workers:
    for row in plan:
        if not row["source"]:
            continue
        for year in range(row["first"].year, row["last"].year + 1):
            tasks = list(year_tasks(row, year))
            print(f"{row['platform']} {row['source']} {year}: {len(tasks)} windows…", flush=True)
            results = list(workers.map(lambda task: get_window(task, mask), tasks))
            frames = [data for data, _ in results if not data.empty]
            yearly = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
            if not yearly.empty:
                yearly = yearly.drop_duplicates(DETECTION_KEY)
                parts.append(yearly)
            kinds = yearly["type"] if not yearly.empty else pd.Series([], dtype="Int8")
            known = int(kinds.notna().sum())
            coverage.append(dict(
                platform=row["platform"], source_product=row["source"], year=year,
                first=tasks[0][2], last=tasks[-1][3], successful_windows=len(tasks),
                downloaded_windows=sum(downloaded for _, downloaded in results),
                detections=len(yearly), known_type=known, unknown_type=len(yearly) - known,
                type_0=int(kinds.eq(0).sum()), type_1=int(kinds.eq(1).sum()),
                type_2=int(kinds.eq(2).sum()), type_3=int(kinds.eq(3).sum()),
                versions=", ".join(sorted(yearly["version"].dropna().unique())) if not yearly.empty else "",
            ))
            print(f"  {len(yearly):,} detections, {known:,} with type", flush=True)
            if known < len(yearly) or yearly.empty:
                print("  Check this year's coverage report for missing types or detections.", flush=True)
coverage_table = pd.DataFrame(coverage)
atomic_path_write(OUTPUT / "firms_coverage_by_year.csv", lambda path: coverage_table.to_csv(path, index=False))
download_complete = True


In [ ]:
# 4. Save the combined catalog and inspect actual coverage.
if not download_complete or not parts:
    raise RuntimeError("No detections were found. See firms_download_plan.csv and the download messages.")

frame = pd.concat(parts, ignore_index=True).sort_values(DETECTION_KEY, ignore_index=True)
frame = frame.drop_duplicates(DETECTION_KEY, ignore_index=True)
front = [*DETECTION_KEY, "frp", "type", "confidence", "source_product"]
frame = frame[[*front, *[column for column in frame if column not in front]]]

path = OUTPUT / "detections_spain.parquet"
atomic_path_write(path, lambda tmp: frame.to_parquet(tmp, index=False, compression="zstd"))
metadata = {
    "source": "NASA FIRMS",
    "products": sorted(frame["source_product"].unique()),
    "requested_dates_utc": [FIRST.isoformat(), LAST.isoformat()],
    "download_plan": [
        {**row, "first": row["first"].isoformat(), "last": row["last"].isoformat()} for row in plan
    ],
    "area": "Spain within the Iberia bounding box",
    "query_bounds_wsen": BBOX,
    "boundary_url": MASK_URL,
    "boundary_sha256": MASK_SHA256,
    "columns": frame.columns.tolist(),
    "rows_by_platform": {name: int(count) for name, count in frame["platform"].value_counts().items()},
    "unknown_type_by_platform": {
        name: int(count) for name, count in frame["type"].isna().groupby(frame["platform"]).sum().items()
    },
    "built_utc": datetime.now(timezone.utc).isoformat(),
}
atomic_path_write(path.with_suffix(".meta.json"),
                  lambda tmp: tmp.write_text(json.dumps(metadata, indent=2), encoding="utf-8"))
print(f"Saved {len(frame):,} detections to {path}")
for name in SOURCES:
    if name not in metadata["rows_by_platform"]:
        print(f"WARNING: no {name} detections in this catalog. See the download plan.")
display(coverage_table.groupby(["platform", "source_product"], as_index=False)[
    ["detections", "known_type", "unknown_type"]
].sum())
